# 7.22 為什麼有些MTP分支串成一條鏈？


平行版本讀到「雨」時，距離 2 的分支直接猜「了」。但「雨停」之後接「了」，和「雨又」之後接「了」，是不同的文字問題。能不能讓較遠的預測沿著中間的 token 一步步往前，而不是每條都只拿同一份雨的特徵？

**DeepSeek-V3 的 MTP** 採用**順序模組**（MTP module）。主模型的核心先讀已知的前文；第一個順序模組再拿到主核心在這個位置的特徵，以及訓練資料裡真正的下一個 token，處理後預測再下一個 token。順序模組是一段額外的神經網路，把兩份資訊合在一起，不是把答案文字直接當成模型的輸出。

沿用句子「雨停了」。讀到「雨」的主核心負責預測「停」；訓練時，第一個順序模組取得「雨」位置的特徵和標準序列中「停」的 embedding 向量，負責預測「了」。若再接第二個順序模組，就取得第一個模組的特徵和「了」的向量，預測〈E〉。下圖示範兩個順序模組，為的是看清資料流。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-7.22-sequential-mtp.svg")))

這份配對規則可以寫成程式：在位置 t，第 k 個順序模組讀進前一段的特徵，以及標準序列的第 t+k 個 token，目標是第 t+k+1 個 token。


In [ ]:
tokens = ["〈B〉", "雨", "停", "了", "〈E〉"]
t = 1  # 主核心讀到「雨」的位置
print(f"主核心：讀 {'／'.join(tokens[: t + 1])} → 目標 {tokens[t + 1]}")
for k in [1, 2]:
    source = "主核心" if k == 1 else f"順序模組{k - 1}"
    given, target = tokens[t + k], tokens[t + k + 1]
    print(f"順序模組{k}：讀{source}的特徵＋標準序列的「{given}」→ 目標 {target}")

`#` 後面的文字是註解，Python 執行時會略過。輸出三行：主核心讀〈B〉／雨、目標是停；順序模組 1 讀主核心的特徵與「停」，目標是了；順序模組 2 讀模組 1 的特徵與「了」，目標是〈E〉。每個模組讀進的標準 token，正好是前一段的目標；它自己的目標永遠在讀進的 token 後面一個位置。`given, target = ...` 一行同時取兩個值，和 [7.10](https://birdhackor.github.io/tiny-perceptron-vlm/7.10.html) 的 `pad_batch` 用逗號一次填三張表是同一種寫法。

為什麼訓練時可以提供「停」？因為第一個順序模組要學的是「雨停之後接什麼」，目標是再下一個的「了」；就像 [7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html) 的對話訓練把正確答案的前半段放進輸入，以便教後半段。這種用資料中的標準序列當訓練前文的做法叫 **teacher forcing**。它本身不等於 MTP：只預測下一個 token 的一般訓練也是這樣做，MTP 額外增加的是從同一個位置出發的更遠目標。

這沒有讓主核心在預測「停」時偷讀「停」：「停」的向量只流向順序模組。也沒有先把「了」交給第一個順序模組，再叫它答「了」。每一條箭頭都要對照「目前讀了什麼、接下來預測什麼」，才能分清合理的前文和洩漏的答案。

實際的 DeepSeek-V3 只用一個順序模組：主模型預測下一個 token，順序模組預測再下一個，不是本圖兩個模組的設定。和 [7.20](https://birdhackor.github.io/tiny-perceptron-vlm/7.20.html) 的平行分支一樣，順序模組也和主模型共用同一個輸出層；DeepSeek-V3 還讓它共用同一張 embedding 表（把 token 查成向量）。輸出層的權重是一張「字表大小×特徵寬度」的大矩陣，每個候選 token 佔一列；兩種做法都共用這張矩陣，所以多加一個預測距離，不必再多存一整張。平行分支與順序模組都屬於 MTP，資料流卻不同。

使用時沒有標準的未來 token 可讀。可以只保留主模型，照原本的方式生成；也可以讓順序模組接上剛猜出的 token，提出後續的候選，再由主模型核對。後者不是把訓練時的標準序列搬到推論，它還需要 [16.14](https://birdhackor.github.io/tiny-perceptron-vlm/16.14.html) 的草稿驗證。

練習把 tokens 改成「〈B〉／雨／又／停／〈E〉」，t 仍是 1，也就是讀到「雨」的位置。執行前先寫下：主核心的目標是什麼？順序模組 1 收到哪個標準 token、目標又是什麼？再執行核對。

<details>
<summary>練習的答案思路</summary>

主核心的目標是「又」；順序模組 1 收到標準序列的「又」，目標是「停」；順序模組 2 收到「停」，目標是〈E〉。每個模組收到的 token 都是前一段的目標。若你畫的是「模組 1 先收到停，再以停當目標」，那就把答案交給了模組自己，不是這裡的配對。

</details>

<details>
<summary>選讀：來源與這張圖的簡化範圍</summary>

[DeepSeek-V3 Technical Report 的第 2.2 節與圖 3](https://arxiv.org/pdf/2412.19437v2)交代順序模組、共用的 embedding 表與輸出層，以及輔助代價；[第 4.2 節](https://arxiv.org/pdf/2412.19437v2)記錄實際只用一個順序模組（D=1）。本圖省略合併特徵用的投影、正規化與模組內部的 Transformer，保留訓練前文和答案的關係。

原文支持這種架構與它實驗條件下的結果，不保證本課的小模型也會改善。架構選擇在[第 14 章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-14.html)展開，成品的取捨見 [19.8](https://birdhackor.github.io/tiny-perceptron-vlm/19.8.html)。

</details>
